# 1. Text Clustering Pipeline

This notebook runs the UMAP + HDBSCAN clustering pipeline on the EMR text fields to group similar faults, and uses Ollama to label the clusters.

In [1]:
import os
import sys
import pandas as pd

# Ultimate foolproof path injection to find 'src' regardless of current working directory
cwd = os.path.abspath(os.getcwd())
project_root = None

# 1. Walk up from current working directory
temp = cwd
for _ in range(4):
    if os.path.exists(os.path.join(temp, 'src', 'config.py')):
        project_root = temp
        break
    parent = os.path.abspath(os.path.join(temp, '..'))
    if parent == temp:
        break
    temp = parent

# 2. Fallback to searching sys.path
if not project_root:
    for path in list(sys.path):
        if not path:
            continue
        candidate = os.path.abspath(path)
        for folder in [candidate, os.path.abspath(os.path.join(candidate, '..')), os.path.abspath(os.path.join(candidate, '..', '..'))]:
            if os.path.exists(os.path.join(folder, 'src', 'config.py')):
                project_root = folder
                break
        if project_root:
            break

if project_root:
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    path_prefix = os.path.relpath(project_root, cwd)
    print(f"Project root found: {project_root}")
    print(f"Current working directory: {cwd}")
    print(f"Relative path prefix: {path_prefix}")
else:
    print("Warning: Could not automatically detect project root. Using default path prefix '..'")
    path_prefix = ".."

from src.config import settings
from src.text_clustering import run_clustering_pipeline
from src.transform import load_emr_data

Project root found: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator
Current working directory: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\notebook
Relative path prefix: ..


In [2]:
print("1. Loading Data...")
file_path = os.path.join(path_prefix, settings.data_dir, settings.emr_file_name)
df = load_emr_data(file_path)
print(f"Loaded {len(df)} records.")

1. Loading Data...
[2026-06-12 21:34:24,822] INFO — src.transform — Loading EMR data: Dashboard EMR(report1776669858353).csv
[2026-06-12 21:34:24,839] INFO — src.transform —   UTF-8 decoding failed — falling back to 'latin1' encoding
[2026-06-12 21:34:25,065] INFO — src.transform —   Loaded 20630 rows, 23 columns.
Loaded 20630 records.


In [3]:
print("2. Running Clustering Pipeline...")
def progress(step, total):
    print(f"Step {step}/{total}")

result = run_clustering_pipeline(df, progress_callback=progress)
print(f"\nClustering completed. Found {result.n_clusters} clusters.")

2. Running Clustering Pipeline...
Step 1/6
Step 2/6


d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-06-12 21:34:36,572] INFO — src.text_clustering — Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2 …


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2710.03it/s]


[2026-06-12 21:34:47,740] INFO — src.text_clustering — Embedding 20630 texts …


...

Step 3/6
[2026-06-12 21:40:52,147] INFO — src.text_clustering — UMAP: 384 -> 10 dims ...


d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Step 4/6
[2026-06-12 21:41:29,573] INFO — src.text_clustering — HDBSCAN (min_cluster_size=5, min_samples=2) …
[2026-06-12 21:41:35,509] INFO — src.text_clustering — Found 1085 clusters, 6111 noise (29.6%)
Step 5/6
[2026-06-12 21:41:35,611] INFO — src.text_clustering — Labeling 1/1085 (cluster 0) …
[2026-06-12 21:42:09,181] INFO — src.text_clustering — Cluster 0 -> Swing Circle Issue (0.85)
[2026-06-12 21:42:09,194] INFO — src.text_clustering — Labeling 2/1085 (cluster 1) …
[2026-06-12 21:42:12,634] INFO — src.text_clustering — Cluster 1 -> PC135 Swing Circle (0.85)
[2026-06-12 21:42:12,635] INFO — src.text_clustering — Labeling 3/1085 (cluster 2) …
[2026-06-12 21:42:24,636] INFO — src.text_clustering — Cluster 2 -> Track Frame Reinforcement (0.85)
[2026-06-12 21:42:24,637] INFO — src.text_clustering — Labeling 4/1085 (cluster 3) …
[2026-06-12 21:42:27,649] INFO — src.text_clustering — Cluster 3 -> Track Frame Reinforcement (0.85)
[2026-06-12 21:42:27,650] INFO — src.text_clustering — L

d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[2026-06-13 05:25:48,588] INFO — src.text_clustering — Visualization -> D:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\output\cluster_visualization.html

Clustering completed. Found 1085 clusters.


In [ ]:
print("3. Results Summary...")
print(f"Noise points: {result.noise_count} ({result.noise_pct}%)")
print(f"Interactive visualization saved to: {result.visualization_path}")

print("\nTop clusters:")
for s in result.cluster_summary[:5]:
    print(f"- {s['label']} (Confidence: {s['confidence']:.2f}): {s['size']} records")

3. Results Summary...
Noise points: 6111 (29.62%)
Interactive visualization saved to: D:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\output\cluster_visualization.html

Top clusters:
- Engine Noise (Confidence: 0.85): 170 records
- Track Link Abnormal Worn (Confidence: 0.85): 134 records
- Fan Motor Issues (Confidence: 0.85): 131 records
- Hydraulic Oil Leak (Confidence: 0.85): 102 records
- Starting Motor Issues (Confidence: 0.85): 101 records


: 